# Image Preview Demo

This notebook uses ipywidgets to request an image file and displays it inline.

In [ ]:
# @markdown Run this cell to check this notebook's version.

current_version = "0.0.1"
notebook_name = "Image_Preview"  # Replace with the actual notebook name

# First of all check if ipywidgets is installed, if not through an informative error
try:
    import ipywidgets as widgets
except ImportError:
    raise ImportError("ipywidgets is not installed. Please install it using 'pip install ipywidgets' or 'conda install -c conda-forge ipywidgets'.")

from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
    
    try:
        import yaml
    except ImportError:
        raise ImportError("pyyaml is not installed. Please install it using 'pip install pyyaml' or 'conda install -c conda-forge pyyaml'.")

    import requests
    import base64

    # Create widgets
    github_repo = widgets.Text(
        value="",
        description="*GitHub Repository URL:",
        placeholder="e.g https://github.com/username/repository",
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    github_token = widgets.Password(
        value="",
        description="*Personal Access Token:",
        placeholder="[Disabled] e.g. ghp_XXXXXXXXXXXXXXXXXXXX",
        disabled=True,
        layout=widgets.Layout(width="70%"),
        style={'description_width': '150px'},
    )

    test_button = widgets.Button(
        description="Test Connection",
        button_style='success',
    )

    output_area = widgets.HTML()

    # By default the repository is considered public
    if "repository_is_private" not in globals():
        repository_is_private = False
    elif repository_is_private:
        github_token.disabled = False
        github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"

    def on_test_button_clicked(b):
        global repository_is_private
        
        # Check that the GitHub repository URL is provided
        if not github_repo.value:
            output_area.value = "⚠️ Please provide a GitHub repository URL."
            return
        
        # Get the owner and repo name from the URL
        repo_url = github_repo.value.rstrip('/')
        github_owner, github_repo_name = repo_url.split('/')[-2:]
        
        # Online version checking file path
        version_file_path = "notebooks/notebook_latest_versions.yaml"
        version_url = f"https://api.github.com/repos/{github_owner}/{github_repo_name}/contents/{version_file_path}"

        # Do an initial request to check if the repository is public
        if not repository_is_private:
            version_response = requests.get(version_url)
            if version_response.status_code == 404:
                repository_is_private = True
                github_token.disabled = False
                github_token.placeholder = "e.g. ghp_XXXXXXXXXXXXXXXXXXXX"
                output_area.value = f"⚠️ We have detected that the repository might be private. Please provide a Personal Access Token and click 'Test Connection' again."
                return
        else:
            headers = {"Accept": "application/vnd.github.v3+json"}
            if not github_token.value:
                output_area.value = "⚠️ Personal Access Token is required for private repositories."
                return
            headers["Authorization"] = f"token {github_token.value}"

            version_response = requests.get(version_url, headers=headers)

        # Check the response status
        if version_response.status_code == 200:
            content = version_response.json()['content']
            decoded_content = base64.b64decode(content).decode('utf-8')
            config = yaml.safe_load(decoded_content)
            latest_version = config.get(notebook_name, "")

            output_area.value = (f"<b>Notebook version:</b> `{current_version}`<br>"
                                f"<b>Latest version available:</b> `{latest_version}`<br>")

            if latest_version == "":
                output_area.value += "⚠️ This notebook is not listed in the version file.<br>"
            elif current_version == latest_version:
                output_area.value += "✅ This notebook is up-to-date.<br>"
            else:
                output_area.value += f"⚠️ A new version of this notebook is available."
        else:
            output_area.value += "⚠️ Could not retrieve the version file.<br>"

    test_button.on_click(on_test_button_clicked)
    # Widget layout
    widget_box = widgets.VBox([github_repo, github_token, test_button, output_area])
    display(widget_box)
except ImportError:
    display(widgets.HTML('To check the notebook version, please go to the <a href="../Welcome.ipynb" target="_blank" style="color: #0066cc; text-decoration: underline;">Welcome notebook</a>.'))

In [ ]:
# @markdown Run this cell to import main functionalities of this notebook.
from IPython.display import display
import ipywidgets as widgets
from io import BytesIO
from PIL import Image
import numpy as np
import base64

from labconstrictor_demo.image_utils import ImageReadError, read_image_from_upload
from labconstrictor_demo.widget_utils import create_image_uploader, notify
print("✅ Done!")

In [ ]:
# @markdown Run this cell to create the image uploader widget and display it along with the status and preview outputs.

uploader = create_image_uploader(description="Upload image", tooltip="Pick an image from disk")
status_out = widgets.HTML()
preview_out = widgets.HTML()

def pil_to_preview_data_url(img, max_size=(800, 800)):
    # Convert uncommon/high-bit-depth modes into something browsers can display well.
    if img.mode in ("I;16", "I;16L", "I;16B", "I", "F"):
        arr = np.array(img).astype(np.float32)
        arr_min = arr.min()
        arr_max = arr.max()

        if arr_max > arr_min:
            arr = (arr - arr_min) / (arr_max - arr_min)
        else:
            arr = np.zeros_like(arr)

        arr = (arr * 255).clip(0, 255).astype(np.uint8)
        img = Image.fromarray(arr, mode="L")

    elif img.mode not in ("L", "RGB", "RGBA"):
        img = img.convert("RGB")

    img = img.copy()
    img.thumbnail(max_size)

    buffer = BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
    return f"data:image/png;base64,{encoded}"

def handle_upload(change):
    if change["name"] != "value":
        return

    files = change["new"]
    status_out.value = ""
    preview_out.value = ""

    if not files:
        status_out.value = " ⚠️ Upload cleared."
        return

    try:
        info = read_image_from_upload(files)
    except (ValueError, ImageReadError) as exc:
        status_out.value = f"❌Could not read the image: {exc}"
        return

    status_out.value = f"✅ Loaded {info.name} ({info.width}x{info.height}px, mode {info.mode})."

    try:
        preview_src = pil_to_preview_data_url(info.image)
        preview_out.value = (
            f"<img src='{preview_src}' alt='Image preview' "
            f"style='max-width: 100%; max-height: 400px; border: 1px solid #ddd; border-radius: 6px;'/>"
        )
    except Exception as exc:
        status_out.value = f"⚠️ Loaded the image, but could not build a browser preview: {exc}"

uploader.observe(handle_upload, names="value")
display(widgets.VBox([uploader, status_out, preview_out]))